CLASSIFICAZIONE CON CNN PRE-ADDESTRATE: MOBILENET E RESNET

Immagina di dover imparare a leggere, non inizia inventando un alfabeto da zero. 
Nel Deep Learning facciamo lo stesso con il transfer learning.
Usiamo dei giganti che sanno già guardare il mondo

- MobilNet e ResNet
- Inferenza
- Meccanismo matematico della softmax

Una CNN pre-addestrata ha già imparato da milioni di immagini caratteristiche molto generali come bordi, texture, forme, e strutture più complesse. In Keras, modelli come MobileNet e ResNet possono essere caricati direttamente con pesi pre-addestrati su ImageNet e usati per classificazione, feature, extraction o fine-tuning

MobileNet e ResNet sono modelli/architetture che possono essere addestrati da qualsiasi immagine. Le versioni, di queste architetture, che troviamo in Keraa sono pre-addestrate con ImageNet.
ImageNet è un database di immagini. E' materiale usato per addestrare i motori di MobilNet e ResNet che si trovano preaddestrati su Keras.

Sia MobilNet che ResNet possono essere utilizzati in due modalità:
1) Uso diretto come classificatore ImageNet: carico il modello completo, posso passare una fotografica ed ottenere qualcosa come: golden retriever, gatto, ecc
Con l'opzione include_top=True, mantiene infatti il classificatore ImageNet originale
2) Transfer Learning: per classificare es. carta, sasso, forbice posso utilizzare MobileNet che però non è stata preaddestrata con queste classi. E' comunque importante caricare il modello pre-addestrato, caricando tutto quello che la CNN ha imparato, togli la classificazione originale e aggiunti le tue classificazioni
Questa è la logica del transfer-learning

Cosa ha già imparato la CNN?

Una CNN pre-addestrata non ha semplicemente memorizzato cane, fatto, automobile
I layer iniziali hanno imparato feature generiche, indicativamente bordi, linee, colori, texture.
poi i layer intermedi: forme pattern e parti di oggetti
ed infine i layer finali: feature molto specializzate.

Per questo motivo posso prendere una CNN allenata su ImageNet e riutilizzarla per un problema differente.

Freeze e Fine-Tuning
La strategia normalmente ha due fasi
prima:
    base_model.trainabile=false
    Congeli tutta la CNN
    poi alleni solamente il nuovo classificatore
    se hai abbastanza dati puoi fare fine-tuning
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    poi ricompili con lr molto basso per non distruggere ciò che la CNN ha già imparato
    optimizer = keras.optimizers.Adam(learning_rate=1e-5)

Chi sono questi giganti di cui parliamo

I Giganti del Deep Learning
Sfruttare la conoscenza pre-addestrata
Addestrare una CNN da zero richiede milioni di immagini e settimane di calcolo. Per queto nel 2026 utilizziamo mobelli pre-addestrati.
Questi modelli hanno già imparato a riconoscere forme, texture, e oggetti complessi grazie al dataset ImageNet, che contiene oltre mille classi diverse.
Addestrare una rete moderna da zero porta via tanto tempo, pertanto partire da un modello già addestrato permette di risparmiare tanto tempo
La loro esperienza può essere utilizzata ai nostri casi

Architetture a confronto
Bilanciamento tra profondità e velocità
* Resnet: introduce le connessioni residue per permettere l'addestramento di reti profondissime senza degradazione del segnale
* MobilNet è ottimizzata per l'edge computing grazie alle convoluzioni separabili che riducono drasticamente i parametri.
* I pesi 'ImageNet' rappresentano la 'memoria' della rete, permettendoci di fare inferenza immediata su oggetti comuni.
* L'operazione fondamentale in queste reti è la convoluzione, che estrae feature spaziali via via più astratte dai pixel grezzi.

Keras ci permette di governare queste strutture in modo semplice

Integrazione con Keras 3
Grazie a keras caricare questi modelli è semplice.
Ma ci sono alcune regole da seguire
Quando carichi un modello con include_top_equal true non stai solo caricando un modello, prendi dalle estrazione dei feature al classificatore finale. Puoi cambiare il motore di calcolo senza cambiare una riga di codice.

Ma attenzione, la rete è esigente, prima di darle da mangiare dobbiamo preparare il cibo correttamente.

Normalizzazione dell'Input
La normalizzazione non è un optional
Ogni modello ha requisiti specifici per il formato dell'immagine, solitamente basati su una riso7luzione fissa e un range di valori.
Senza una corretta pre-elaborazione (scaling e centratura), la rete interpreterà i dati in modo errato producendo risultati inattendibili.

Ogni modello ha la sua tipologia di dati che si aspetta in input.

Una volta normalizzati i dati entriamo nella fase dell'inferenza rapida

Inferenza Fast-Frame
L'arte della velocità
L'inferenza è il momento della verità, il passaggio del dato attraverso tutti i layer della rete per ottenere la risposta.
L'inferenza è il processo in cui un modello già addestrato effettua una predizoine su un nuovo dato mai visto in precedenza.
Per applicazioni video, dobbiamo garantire che questo processo avvenga in pochi millisencondi per evitare lag nello strem.

Il nostro obbiettivo è puntare alla inferenza veloce.

Flusso di Predizione
Dall'array al risultato
- Il caricamente del frame deve essere seguito da un resizing bilineare per adattarsi alla input_shape del modello
- L'aggiunta della dimensione batch è obbligatoria, trasformando l'immagine singola in un tensore di rango quattro. Anche se processiamo una singola immagine, la rete di aspetta una lista.
- La classe predetta è quella associata all'indice del valore massimo nel vettore di probabilità (Argmax)
- La latenza di inferenza dipende dalla complessità del modello: MobilNet è circa 10 volte più veloce di ResNet su CPU standard.

Ma come possiamo spingere queste prestazioni ancora più in la?

Ottimizzazione HardWare

Se la velocità di base non basta abbiamo dei trucchi del mesteire
primo di tutti l'accellerazione GPU
Poi anche la quantizzazione, possiamo comprimere i pesi della rete.Usiamo piccoli numeri interi, int8

Interpretare la Mente Digitale

L'output della rete è un paradosso, noi vorremmo: 'E' un gatto' ma la rete ci restituisce un vettore di 1.000 numeri decimali
Dobbiamo usare la statistica per trasformare questi punteggi di attivazione in una probabiltà comprensibile per l'essere umano.
Questo vettore rappresenta l'attivazione degli ultimi neuroni.
Lo strumento per far generare questo vettore è la softmax


In [ ]:
import os
import requests
import numpy as np
import cv2
from io import BytesIO

# --- CONFIGURAZIONE BACKEND 2026 ---
# Impostiamo PyTorch come motore di calcolo prima dell'import di Keras.
os.environ["KERAS_BACKEND"] = "torch"

import keras

class ImageClassifierSOTA:
    """
    Wrapper per la classificazione d'immagine.
    Integra MobileNetV2 con pre-processing dinamico da URL web.
    """
    def __init__(self, model_name="MobileNetV2"):
        print(f"Inizializzazione modello: {model_name} con pesi ImageNet...")
        
        # Carichiamo MobileNetV2: architettura ottimizzata per l'efficienza.
        # Usa le 'Depthwise Separable Convolutions' per ridurre i parametri senza perdere troppa accuratezza.
        self.model = keras.applications.MobileNetV2(
            weights="imagenet", 
            include_top=True
        )
        
        # Dimensione standard per MobileNetV2: 224x224 pixel.
        self.input_shape = (224, 224)

    def download_image(self, url):
        """
        Scarica un'immagine da un URL e la converte in formato OpenCV (BGR).
        """
        print(f"Scaricamento immagine da: {url}...")
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            
            # Conversione dei byte in array numpy per OpenCV
            image_bytes = np.asarray(bytearray(response.content), dtype=np.uint8)
            img = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)
            
            if img is None:
                raise ValueError("Il file scaricato non è un'immagine valida.")
            return img
        except Exception as e:
            print(f"Errore durante il download: {e}")
            return None

    def preprocess(self, img_bgr):
        """
        Trasforma l'immagine grezza in un tensore pronto per il Deep Learning.
        """
        # 1. Conversione BGR -> RGB (Cruciale per i modelli addestrati su ImageNet)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # 2. Resize: Adattamento alla risoluzione 224x224
        img_resized = cv2.resize(img_rgb, self.input_shape)
        
        # 3. Espansione Batch: Da (224, 224, 3) a (1, 224, 224, 3)
        img_batch = np.expand_dims(img_resized, axis=0).astype("float32")
        
        # 4. Normalizzazione: MobileNetV2 richiede pixel nel range [-1, 1]
        return keras.applications.mobilenet_v2.preprocess_input(img_batch)

    def classify(self, preprocessed_img, top_k=3):
        """
        Esegue l'inferenza e decodifica i risultati in classi leggibili.
        """
        # Inferenza tramite il backend selezionato (PyTorch)
        preds = self.model(preprocessed_img, training=False)
        
        # Spostiamo il risultato su CPU e convertiamo in Numpy per la decodifica
        preds_numpy = keras.ops.convert_to_numpy(preds)
        
        # Decode: Converte i vettori di probabilità in (ID, Etichetta, Probabilità)
        return keras.applications.mobilenet_v2.decode_predictions(preds_numpy, top=top_k)[0]

# --- WORKFLOW DI ESECUZIONE ---
if __name__ == "__main__":
    # 1. Istanza del classificatore
    classifier = ImageClassifierSOTA()

    # 2. URL di test (Esempio: un Golden Retriever da Unsplash)
    test_url = "https://images.unsplash.com/photo-1552053831-71594a27632d?q=80&w=800"

    try:
        # Recupero immagine
        raw_img = classifier.download_image(test_url)
        
        if raw_img is not None:
            # Pre-processing
            processed_data = classifier.preprocess(raw_img)

            # Classificazione
            predictions = classifier.classify(processed_data)

            print("\n" + "="*40)
            print("   RISULTATI CLASSIFICAZIONE SOTA")
            print("="*40)
            for i, (imagenet_id, label, prob) in enumerate(predictions):
                print(f"{i+1}. {label.upper():<20} {prob*100:>6.2f}%")
            print("="*40)

            # Visualizzazione rapida (facoltativa)
            cv2.putText(raw_img, f"Pred: {predictions[0][1]}", (20, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.imshow("Classificazione Web", raw_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        
    except Exception as e:
        print(f"Errore critico nella pipeline: {e}")